# Day 01: LLM Inference Mechanics
> *Inference Engineering* — Chapter 2.2 | Philip Kiely, Baseten Books 2026

**Layer:** Runtime  
**Covers:** Tokenization · KV Cache · Autoregressive Decoding · Sampling Strategies

## Concept Overview

LLMs are **autoregressive token generation models**: they generate new tokens one at a time, each based on every previous token. A *token* is the atomic unit of a language model — a number that represents a chunk of text (typically a word or fraction of a word), produced by *subword tokenization*.

Inference has two primary phases:
1. **Prefill** — process the entire input sequence in parallel, computing attention for every input token and storing those key-value pairs in a *KV cache*.
2. **Decode** — perform sequential forward passes, one token at a time, looking up the KV cache to avoid recomputing prior attention.

The two most expensive steps — building the KV cache during prefill and generating logit vectors during decode — consume the overwhelming majority of inference time and GPU memory.

In [ ]:
# Setup: all standard-library or pre-installed packages on DGX Spark nodes
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import time
import math
import random

# Optional: torch for GPU-backed tensor ops
try:
    import torch
    TORCH_AVAILABLE = True
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"PyTorch {torch.__version__} | device: {DEVICE}")
    if DEVICE == 'cuda':
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
except ImportError:
    TORCH_AVAILABLE = False
    DEVICE = 'cpu'
    print("PyTorch not available — running numpy-only mode")

%matplotlib inline
plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor'] = '#161b22'
plt.rcParams['axes.edgecolor'] = '#30363d'
plt.rcParams['text.color'] = '#e6edf3'
plt.rcParams['axes.labelcolor'] = '#e6edf3'
plt.rcParams['xtick.color'] = '#8b949e'
plt.rcParams['ytick.color'] = '#8b949e'
plt.rcParams['grid.color'] = '#21262d'

: 

## Part 1: Tokenization

A *tokenizer* is a simple mapping between strings and their numerical token representations — no neural network required. The *vocabulary* is the complete set of all (token_id → string) mappings a model knows.

Modern LLMs use **subword tokenization**: common words get a single token, while rare words are split into multiple tokens. The sentence "In ference engineering makes AI apps fast ." tokenizes as:

> Book **Figure 2.5** shows: `[644, 2251, 15009, 3727, 15592, 10721, 5043, 13]` — 8 tokens for 6 words + punctuation.

**Why it matters for inference:** Fewer tokens = faster end-to-end latency. If a model uses a more efficient tokenizer that represents the same text in 20% fewer tokens, it runs 20% faster — before any GPU optimization.

In [ ]:
# Simulate a minimal BPE-style tokenizer from scratch
# We'll build a toy vocabulary and demonstrate tokenization mechanics

# Toy vocabulary: word → token_id
VOCAB = {
    '<pad>': 0, '<bos>': 1, '<eos>': 2, '<unk>': 3,
    # Common subwords
    'In': 644, '▁ference': 2251, '▁engineering': 15009,
    '▁makes': 3727, '▁AI': 15592, '▁apps': 10721,
    '▁fast': 5043, '.': 13,
    # Additional words for our demo
    '▁inference': 100, '▁is': 200, '▁slow': 300,
    '▁without': 400, '▁KV': 500, '▁cache': 600,
    '▁The': 700, '▁model': 800, '▁gener': 900,
    'ates': 901, '▁tokens': 1000, '▁one': 1100,
    '▁at': 1200, '▁a': 1300, '▁time': 1400,
}
ID_TO_TOKEN = {v: k for k, v in VOCAB.items()}

def tokenize(text: str) -> list[int]:
    """Greedy longest-match tokenizer (simplified BPE)."""
    # In real BPE, we'd apply merge rules. Here we do word-level splitting
    # with a '▁' sentinel for space-prefixed words (SentencePiece convention)
    tokens = []
    words = text.split()
    for i, word in enumerate(words):
        # Add space sentinel for non-first words
        key = ('▁' + word) if i > 0 else word
        if key in VOCAB:
            tokens.append(VOCAB[key])
        elif word in VOCAB:
            tokens.append(VOCAB[word])
        else:
            # Fall back to character-level splits + <unk>
            tokens.append(VOCAB['<unk>'])
    return tokens

def detokenize(token_ids: list[int]) -> str:
    """Convert token IDs back to text."""
    pieces = [ID_TO_TOKEN.get(tid, '<unk>') for tid in token_ids]
    # Remove '▁' sentinels and join
    return ' '.join(p.lstrip('▁') for p in pieces)

# --- Demo ---
sentence = "Inference engineering makes AI apps fast ."
token_ids = tokenize(sentence)

print(f"Input:     '{sentence}'")
print(f"Token IDs: {token_ids}")
print(f"Decoded:   '{detokenize(token_ids)}'")
print(f"\nVocab size (toy): {len(VOCAB)}")
print(f"Real LLMs typically have: 32k–200k tokens in vocabulary")
print(f"\nTokenization breakdown:")
for tok_id in token_ids:
    print(f"  {tok_id:6d}  →  '{ID_TO_TOKEN[tok_id]}'")

In [ ]:
# Visualize token efficiency: same concept, different tokenizer efficiency
# Demonstrates why fewer tokens = faster inference

EXAMPLES = [
    # (text, efficient_tokens, inefficient_tokens)
    ("Hello world",                        2,  3),
    ("LLM inference",                      2,  4),
    ("The quick brown fox",                4,  6),
    ("Autoregressive token generation",    3,  7),
    ("Mixture of Experts architecture",    4,  8),
    ("PagedAttention memory management",   3,  7),
    ("Flash attention CUDA kernel",        4,  7),
]

texts = [e[0] for e in EXAMPLES]
efficient = [e[1] for e in EXAMPLES]
inefficient = [e[2] for e in EXAMPLES]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(texts))
w = 0.35

bars1 = ax.bar(x - w/2, efficient, w, label='Efficient tokenizer', color='#238636', alpha=0.9)
bars2 = ax.bar(x + w/2, inefficient, w, label='Inefficient tokenizer', color='#da3633', alpha=0.9)

ax.set_ylabel('Token count', color='#e6edf3')
ax.set_title('Token Efficiency: fewer tokens = faster inference\n(same text, different vocabularies)', 
             color='#e6edf3', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels([t[:20] + '…' if len(t) > 20 else t for t in texts], 
                   rotation=30, ha='right', fontsize=8)
ax.legend(framealpha=0.2)
ax.grid(axis='y', alpha=0.3)

# Annotate savings
for i, (eff, ineff) in enumerate(zip(efficient, inefficient)):
    savings_pct = (ineff - eff) / ineff * 100
    ax.annotate(f'-{savings_pct:.0f}%', xy=(i - w/2, eff + 0.1), 
                ha='center', va='bottom', fontsize=7, color='#3fb950')

plt.tight_layout()
plt.show()
print(f"Average token reduction: {np.mean([(i-e)/i*100 for e,i in zip(efficient,inefficient)]):.1f}%")

## Part 2: The Prefill/Decode Split

After tokenization (step zero), inference has two primary phases:

| Phase | What happens | Bottleneck |
|-------|-------------|------------|
| **Prefill** | All input tokens processed *in parallel* through every transformer block; K and V tensors stored in KV cache | Compute-bound (large matmuls) |
| **Decode** | One new token generated per forward pass; KV cache read for attention | Memory-bandwidth-bound (small matmuls, large cache reads) |

This asymmetry — prefill is compute-bound, decode is memory-bandwidth-bound — drives many inference engineering decisions: batching strategies, hardware selection, disaggregation (covered in Day 16).

**Book Figure 2.6** shows the decode loop: vocabulary → logits → probabilities → next token → repeat.

In [ ]:
# Simulate the prefill vs decode time profile
# Models: prefill processes all tokens at once, decode is sequential

def simulate_inference_phases(
    prompt_tokens: int,
    output_tokens: int,
    prefill_ms_per_token: float = 0.5,   # fast: parallel
    decode_ms_per_token: float = 15.0,   # slower: sequential + memory reads
):
    """Simulate time breakdown for a single inference request."""
    prefill_time = prompt_tokens * prefill_ms_per_token
    decode_time = output_tokens * decode_ms_per_token
    total = prefill_time + decode_time
    
    return {
        'prefill_ms': prefill_time,
        'decode_ms': decode_time,
        'total_ms': total,
        'ttft_ms': prefill_time,                        # Time To First Token
        'tpot_ms': decode_ms_per_token,                 # Time Per Output Token
        'throughput_tok_s': 1000 / decode_ms_per_token, # decode throughput
    }

# Run scenarios
scenarios = [
    ("Short prompt, short output",   128,  64),
    ("Long prompt, short output",   4096,  64),
    ("Short prompt, long output",    128, 512),
    ("Long prompt, long output",    4096, 512),
]

print(f"{'Scenario':<35} {'Prefill':>10} {'Decode':>10} {'Total':>10} {'TTFT':>10}")
print("-" * 80)
for name, prompt_toks, out_toks in scenarios:
    r = simulate_inference_phases(prompt_toks, out_toks)
    print(f"{name:<35} {r['prefill_ms']:>8.0f}ms {r['decode_ms']:>8.0f}ms "
          f"{r['total_ms']:>8.0f}ms {r['ttft_ms']:>8.0f}ms")

# Visualization: stacked bar of prefill vs decode time
fig, ax = plt.subplots(figsize=(10, 4))
names = [s[0] for s in scenarios]
results = [simulate_inference_phases(s[1], s[2]) for s in scenarios]
prefills = [r['prefill_ms'] for r in results]
decodes  = [r['decode_ms']  for r in results]

x = np.arange(len(names))
ax.bar(x, prefills, label='Prefill (compute-bound)', color='#1f6feb', alpha=0.9)
ax.bar(x, decodes, bottom=prefills, label='Decode (memory-bw-bound)', color='#388bfd', alpha=0.7)

ax.set_ylabel('Time (ms)', color='#e6edf3')
ax.set_title('Prefill vs Decode Time Breakdown', color='#e6edf3', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right', fontsize=9)
ax.legend(framealpha=0.2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Part 3: The KV Cache

**Attention** is the mechanism transformers use to relate a given token to all other tokens in the sequence. The standard form is **scaled dot-product attention** (from "Attention Is All You Need", Vaswani et al. 2017):

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Where:
- **Q (queries):** The embedded representation of the token being generated or updated
- **K (keys):** Representations of all prior tokens  
- **V (values):** Computed attention values for all prior tokens

Because attention looks at every prior token, naively it's $O(n^2)$ with sequence length. The **KV cache** makes it $O(n)$ in practice: K and V tensors for all prior tokens are computed once (during prefill) and stored in GPU memory, so decode only needs to compute Q for the new token and do a single lookup.

> Book Figure 2.8 shows the attention equation. The $\sqrt{d_k}$ scaling factor prevents dot products from growing so large that softmax saturates into near-zero gradients.

**KV cache memory cost:**  
$$\text{KV memory} = 2 \times n_{layers} \times n_{heads} \times d_{head} \times \text{seq\_len} \times \text{bytes\_per\_element}$$

In [ ]:
# Implement scaled dot-product attention from scratch
# Then show why the KV cache is essential

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V
    
    Q: (seq_len, d_k)  — queries
    K: (seq_len, d_k)  — keys
    V: (seq_len, d_v)  — values
    mask: (seq_len, seq_len) — causal mask (upper triangle = -inf)
    """
    d_k = Q.shape[-1]
    
    # Step 1: dot product of queries and keys
    scores = Q @ K.T  # (seq_len, seq_len)
    
    # Step 2: scale by sqrt(d_k) to keep gradients stable
    scores = scores / math.sqrt(d_k)
    
    # Step 3: apply causal mask (LLMs can't look at future tokens)
    if mask is not None:
        scores = scores + mask  # mask has -inf where future tokens are
    
    # Step 4: softmax to get attention weights (sum to 1 per row)
    # Stable softmax: subtract max before exp
    scores_max = scores.max(axis=-1, keepdims=True)
    exp_scores = np.exp(scores - scores_max)
    weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)  # (seq_len, seq_len)
    
    # Step 5: weighted sum of values
    output = weights @ V  # (seq_len, d_v)
    
    return output, weights

# Demo with small dimensions
np.random.seed(42)
seq_len = 6
d_k = 8   # key/query dimension
d_v = 8   # value dimension

Q = np.random.randn(seq_len, d_k) * 0.5
K = np.random.randn(seq_len, d_k) * 0.5
V = np.random.randn(seq_len, d_v) * 0.5

# Causal mask: position i can only attend to positions <= i
causal_mask = np.triu(np.full((seq_len, seq_len), -np.inf), k=1)

output, weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot attention weights (heatmap)
im1 = axes[0].imshow(weights, cmap='viridis', vmin=0, vmax=1)
axes[0].set_title('Causal Attention Weights\n(each row sums to 1)', color='#e6edf3')
axes[0].set_xlabel('Key position (what we attend TO)', color='#e6edf3')
axes[0].set_ylabel('Query position (token being generated)', color='#e6edf3')
plt.colorbar(im1, ax=axes[0])

# The upper triangle is zero — causal masking prevents future-token attention
axes[0].text(3.5, 1.5, 'Masked\n(future)', ha='center', va='center', 
             color='white', fontsize=9, alpha=0.7)

# Plot unscaled vs scaled scores to show why sqrt(d_k) matters
test_Q = np.random.randn(1, 64) * 0.5   # d_k = 64
test_K = np.random.randn(100, 64) * 0.5

raw_scores    = (test_Q @ test_K.T).flatten()
scaled_scores = (test_Q @ test_K.T / math.sqrt(64)).flatten()

def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

axes[1].plot(softmax(raw_scores),    label=f'Unscaled  (std={raw_scores.std():.2f})',    color='#da3633', alpha=0.8)
axes[1].plot(softmax(scaled_scores), label=f'Scaled ÷√{64} (std={scaled_scores.std():.2f})', color='#3fb950', alpha=0.8)
axes[1].set_title('Effect of √d_k Scaling on Attention\n(d_k=64)', color='#e6edf3')
axes[1].set_xlabel('Key position', color='#e6edf3')
axes[1].set_ylabel('Softmax attention weight', color='#e6edf3')
axes[1].legend(framealpha=0.2)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Without scaling: attention collapses to argmax (one token gets all weight)")
print(f"With √d_k scaling: smoother distribution, all tokens contribute")

In [ ]:
# KV Cache memory calculator
# This is real math you'd use when sizing GPU memory for a deployment

def kv_cache_memory_gb(
    num_layers: int,
    num_kv_heads: int,
    head_dim: int,
    seq_len: int,
    bytes_per_element: int = 2,  # FP16 = 2 bytes, FP8 = 1 byte
    batch_size: int = 1,
) -> float:
    """
    KV cache = 2 (K and V) × layers × heads × head_dim × seq_len × bytes
    Factor of 2: one tensor for K, one for V.
    """
    total_bytes = (
        2            # K and V
        * num_layers
        * num_kv_heads
        * head_dim
        * seq_len
        * bytes_per_element
        * batch_size
    )
    return total_bytes / 1e9

# Common model configs (approximate)
MODELS = {
    'Llama-3.1-8B':      dict(num_layers=32, num_kv_heads=8,  head_dim=128),
    'Llama-3.1-70B':     dict(num_layers=80, num_kv_heads=8,  head_dim=128),
    'Qwen2.5-7B':        dict(num_layers=28, num_kv_heads=4,  head_dim=128),
    'Mistral-7B':        dict(num_layers=32, num_kv_heads=8,  head_dim=128),
    'DeepSeek-R1-8B':    dict(num_layers=32, num_kv_heads=8,  head_dim=128),
}

SEQ_LENS = [1024, 4096, 16384, 65536, 131072]

print(f"{'Model':<22} {'1K toks':>10} {'4K toks':>10} {'16K toks':>10} {'64K toks':>10} {'128K toks':>10}")
print("-" * 75)
for model_name, cfg in MODELS.items():
    sizes = [kv_cache_memory_gb(**cfg, seq_len=s) for s in SEQ_LENS]
    row = f"{model_name:<22}"
    for s in sizes:
        row += f" {s:>8.2f}GB"
    print(row)

print("\nNote: GQA (Grouped Query Attention) reduces KV heads vs attention heads.")
print("Llama-3.1-70B has 64 attention heads but only 8 KV heads (8x memory reduction).")
print("\nDGX Spark has 128GB unified memory — KV cache for a 128K context of Llama-70B:")
gb_70b_128k = kv_cache_memory_gb(num_layers=80, num_kv_heads=8, head_dim=128, seq_len=131072)
print(f"  {gb_70b_128k:.1f} GB — before the model weights (~140 GB in FP16)")

## Part 4: Autoregressive Decoding

Each forward pass in the decode phase:
1. Takes the last generated token as input
2. Looks up K, V for all prior tokens from the KV cache
3. Runs the new token through all transformer blocks
4. The **LMHead** (output layer) projects hidden state → logit vector of shape `[vocab_size]`
5. Logits are normalized via softmax to give a probability distribution over the vocabulary
6. One token is **sampled** from that distribution
7. Repeat until stop token (`<eos>`) or `max_tokens` limit

**Sampling strategies** control the quality/diversity tradeoff:
- **Temperature** $T$: scales logits before softmax — $\text{logits} \leftarrow \text{logits} / T$. Lower $T$ → more deterministic.
- **Top-k**: keep only the $k$ highest-probability tokens, re-normalize.
- **Top-p** (nucleus): keep the smallest set of tokens whose cumulative probability ≥ $p$.

In [ ]:
# Implement the full token sampling pipeline
# This is exactly what inference engines do at the end of every forward pass

def softmax_temperature(logits: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    """Apply temperature scaling then softmax."""
    if temperature == 0:
        # Greedy: argmax (deterministic)
        probs = np.zeros_like(logits, dtype=float)
        probs[np.argmax(logits)] = 1.0
        return probs
    scaled = logits / temperature
    exp_s = np.exp(scaled - scaled.max())  # stable softmax
    return exp_s / exp_s.sum()

def top_k_filter(probs: np.ndarray, k: int) -> np.ndarray:
    """Zero out all but the top-k probability tokens, then renormalize."""
    if k <= 0 or k >= len(probs):
        return probs
    threshold = np.sort(probs)[-k]  # k-th largest value
    filtered = np.where(probs >= threshold, probs, 0.0)
    return filtered / filtered.sum()

def top_p_filter(probs: np.ndarray, p: float) -> np.ndarray:
    """Keep the smallest set of tokens whose cumulative prob >= p (nucleus sampling)."""
    sorted_idx = np.argsort(probs)[::-1]          # descending order
    cumsum = np.cumsum(probs[sorted_idx])
    cutoff_idx = np.searchsorted(cumsum, p) + 1   # +1 to include the crossing token
    
    # Keep only the nucleus tokens
    keep_idx = sorted_idx[:cutoff_idx]
    filtered = np.zeros_like(probs)
    filtered[keep_idx] = probs[keep_idx]
    return filtered / filtered.sum()

def sample_token(logits: np.ndarray, temperature: float = 1.0, 
                 top_k: int = 0, top_p: float = 1.0) -> int:
    """Full sampling pipeline: temperature → top-k → top-p → multinomial sample."""
    probs = softmax_temperature(logits, temperature)
    probs = top_k_filter(probs, top_k)
    probs = top_p_filter(probs, top_p)
    return int(np.random.choice(len(probs), p=probs))

# --- Visualize sampling strategies ---
np.random.seed(123)
VOCAB_SIZE = 30
# Create realistic-looking logits: a few high-prob tokens, long tail
raw_logits = np.random.randn(VOCAB_SIZE) * 2
raw_logits[3] = 5.5   # "Abs" - most likely
raw_logits[7] = 4.2   # "Act" - second most likely
raw_logits[12] = 3.1  # third candidate

STRATEGIES = [
    ('Greedy (T=0)',          dict(temperature=0,    top_k=0,  top_p=1.0)),
    ('T=0.3 (conservative)',  dict(temperature=0.3,  top_k=0,  top_p=1.0)),
    ('T=1.0 (default)',       dict(temperature=1.0,  top_k=0,  top_p=1.0)),
    ('T=1.5 (creative)',      dict(temperature=1.5,  top_k=0,  top_p=1.0)),
    ('Top-k=5, T=1.0',        dict(temperature=1.0,  top_k=5,  top_p=1.0)),
    ('Top-p=0.9, T=1.0',      dict(temperature=1.0,  top_k=0,  top_p=0.9)),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
axes = axes.flatten()

for ax, (name, kwargs) in zip(axes, STRATEGIES):
    probs = softmax_temperature(raw_logits, kwargs['temperature'])
    probs = top_k_filter(probs, kwargs['top_k'])
    probs = top_p_filter(probs, kwargs['top_p'])
    
    n_active = (probs > 0).sum()
    colors = ['#238636' if p > 0 else '#21262d' for p in probs]
    ax.bar(range(VOCAB_SIZE), probs, color=colors, alpha=0.85)
    ax.set_title(f'{name}\n({n_active} active tokens)', color='#e6edf3', fontsize=9)
    ax.set_xlabel('Token ID', color='#8b949e', fontsize=8)
    ax.set_ylabel('P(token)', color='#8b949e', fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Sampling Strategy Comparison — same logits, different outcomes', 
             color='#e6edf3', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Full autoregressive decode loop simulation
# This is the inference engine's inner loop, stripped to essentials

class ToyLLM:
    """
    Minimal LLM simulator to demonstrate the prefill/decode loop.
    Uses a fixed transition table instead of real weights — the mechanics
    (KV cache, autoregressive generation, sampling) are identical.
    """
    VOCAB = ['<bos>', '<eos>', 'The', 'model', 'generates', 'tokens',
             'one', 'at', 'a', 'time', 'using', 'KV', 'cache', 'fast',
             'slow', 'memory', 'GPU', 'inference', 'is', 'efficient']
    
    def __init__(self, d_model=16, n_layers=2):
        self.vocab_size = len(self.VOCAB)
        self.d_model = d_model
        self.n_layers = n_layers
        self.tok2id = {w: i for i, w in enumerate(self.VOCAB)}
        self.id2tok = {i: w for w, i in self.tok2id.items()}
        
        # Fixed "language model" logit biases — makes generation readable
        np.random.seed(7)
        self.transition = np.random.randn(self.vocab_size, self.vocab_size) * 0.3
        # Wire in some reasonable transitions
        for tok, next_toks in [
            ('The', ['model', 'GPU', 'KV']),
            ('model', ['generates', 'is']),
            ('generates', ['tokens', 'one']),
            ('tokens', ['one', 'fast', 'using']),
            ('one', ['at', 'token']),
            ('at', ['a']),
            ('a', ['time']),
            ('time', ['using', '<eos>']),
            ('using', ['KV', 'GPU']),
            ('KV', ['cache']),
            ('cache', ['fast', 'is', '<eos>']),
        ]:
            if tok in self.tok2id:
                for nt in next_toks:
                    if nt in self.tok2id:
                        self.transition[self.tok2id[tok], self.tok2id[nt]] += 3.0
        
        # KV cache: stores (K, V) per layer per token position
        # Shape: n_layers × seq_len × d_model (simplified — real shape is per-head)
        self.kv_cache_k = []  # list of (n_layers, d_model) arrays
        self.kv_cache_v = []
    
    def prefill(self, prompt_ids: list[int]) -> np.ndarray:
        """Process all prompt tokens in parallel. Populate KV cache."""
        self.kv_cache_k = []
        self.kv_cache_v = []
        
        # In reality: run all tokens through transformer blocks in one big batch
        # Here: just store fake K, V tensors for each position
        for pos, tok_id in enumerate(prompt_ids):
            k = np.random.randn(self.n_layers, self.d_model) * 0.1  # simulated
            v = np.random.randn(self.n_layers, self.d_model) * 0.1
            self.kv_cache_k.append(k)
            self.kv_cache_v.append(v)
        
        # Return logits for the last prompt token (used to generate first output token)
        return self.transition[prompt_ids[-1]].copy()
    
    def decode_step(self, tok_id: int) -> np.ndarray:
        """One decode step: update KV cache, compute logits for next token."""
        # In reality: tok_id goes through embedding → attention (using KV cache) → FFN → LMHead
        # Here: just append to KV cache and look up transition probabilities
        k = np.random.randn(self.n_layers, self.d_model) * 0.1
        v = np.random.randn(self.n_layers, self.d_model) * 0.1
        self.kv_cache_k.append(k)
        self.kv_cache_v.append(v)
        
        # Logits from last token transition
        return self.transition[tok_id].copy()
    
    def generate(self, prompt: str, max_tokens: int = 12,
                 temperature: float = 0.8, top_k: int = 5) -> str:
        """Full generation loop: tokenize → prefill → decode loop."""
        # Tokenize prompt
        prompt_ids = [self.tok2id.get(w, 0) for w in prompt.split()]
        prompt_ids = [1] + prompt_ids  # prepend <bos>
        
        print(f"  Prefill: {len(prompt_ids)} tokens → building KV cache...")
        logits = self.prefill(prompt_ids)
        print(f"  KV cache size after prefill: {len(self.kv_cache_k)} positions")
        
        generated = []
        cur_tok = prompt_ids[-1]
        
        print(f"  Decode loop:")
        for step in range(max_tokens):
            next_tok = sample_token(logits, temperature=temperature, top_k=top_k)
            tok_str = self.id2tok[next_tok]
            kv_len = len(self.kv_cache_k)
            print(f"    step {step+1:2d}: sampled '{tok_str:12s}' | KV cache: {kv_len} → {kv_len+1} positions")
            
            if tok_str == '<eos>':
                print(f"    → stop token generated, halting")
                break
            
            generated.append(tok_str)
            logits = self.decode_step(next_tok)
        
        return ' '.join(prompt.split() + generated)

# Run a generation
print("=" * 55)
print("Autoregressive Generation Demo")
print("=" * 55)
llm = ToyLLM()
np.random.seed(42)
result = llm.generate("The model", max_tokens=10, temperature=0.7, top_k=4)
print(f"\nFinal output: '{result}'")

## Part 5: KV Cache Growth & Memory Pressure During Generation

The KV cache is **built incrementally**: it grows by one position every decode step. This has critical implications:

- **Memory grows with sequence length** — long outputs eat into the GPU memory budget
- **Decode slows as cache grows** — more keys and values to read from HBM per step
- **Context window is a hard limit** — when the cache is full, the request fails or the oldest entries must be evicted

This is exactly why **PagedAttention** (vLLM, Day 07) and **Prefix Caching** (Day 14) exist — to manage KV cache memory more efficiently.

In [ ]:
# Visualize KV cache growth and memory pressure during a generation
# Also measure the actual attention computation time vs sequence length

def attention_time_vs_seqlen(seq_lengths, d_k=128, n_heads=8, n_trials=3):
    """Measure how attention compute scales with sequence length."""
    times = []
    for seq_len in seq_lengths:
        trial_times = []
        for _ in range(n_trials):
            Q = np.random.randn(1, d_k)         # single query (decode step)
            K = np.random.randn(seq_len, d_k)   # all keys in KV cache
            V = np.random.randn(seq_len, d_k)   # all values in KV cache
            
            t0 = time.perf_counter()
            # This is what happens every decode step:
            scores = Q @ K.T / math.sqrt(d_k)    # (1, seq_len)
            weights = softmax(scores.flatten())   
            output = weights @ V                  # (d_k,) — weighted sum of values
            t1 = time.perf_counter()
            trial_times.append((t1 - t0) * 1000)
        times.append(np.mean(trial_times))
    return times

seq_lengths = [64, 128, 256, 512, 1024, 2048, 4096, 8192]
attn_times  = attention_time_vs_seqlen(seq_lengths)

# KV cache memory at each sequence length (Llama-3-8B config)
kv_mems_gb = [kv_cache_memory_gb(num_layers=32, num_kv_heads=8, 
                                   head_dim=128, seq_len=s) for s in seq_lengths]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Attention time vs sequence length
ax1.plot(seq_lengths, attn_times, 'o-', color='#1f6feb', linewidth=2, markersize=6)
# Fit: should be linear since it's Q@K.T with single query row
coeffs = np.polyfit(seq_lengths, attn_times, 1)
fit_line = np.poly1d(coeffs)(seq_lengths)
ax1.plot(seq_lengths, fit_line, '--', color='#3fb950', alpha=0.7, label=f'Linear fit')
ax1.set_xlabel('KV cache length (tokens in context)', color='#e6edf3')
ax1.set_ylabel('Attention time per decode step (ms)', color='#e6edf3')
ax1.set_title('Decode Attention Time vs Context Length\n(single-query, CPU numpy)', color='#e6edf3')
ax1.legend(framealpha=0.2)
ax1.grid(alpha=0.3)
ax1.set_xscale('log', base=2)

# KV cache memory vs seq length
ax2.fill_between(seq_lengths, kv_mems_gb, alpha=0.4, color='#da3633')
ax2.plot(seq_lengths, kv_mems_gb, 'o-', color='#da3633', linewidth=2, markersize=6)
ax2.axhline(y=80, color='#f0883e', linestyle='--', alpha=0.8, label='80GB GPU (A100)')
ax2.axhline(y=128, color='#3fb950', linestyle='--', alpha=0.8, label='128GB (DGX Spark)')
ax2.set_xlabel('Sequence length (tokens)', color='#e6edf3')
ax2.set_ylabel('KV Cache memory (GB)', color='#e6edf3')
ax2.set_title('KV Cache Memory Growth — Llama-3-8B\n(FP16, batch=1)', color='#e6edf3')
ax2.legend(framealpha=0.2)
ax2.grid(alpha=0.3)
ax2.set_xscale('log', base=2)

plt.tight_layout()
plt.show()

print("Attention is LINEAR in seq_len for a single decode step (one query, N keys).")
print("It would be QUADRATIC if we recomputed all K,V from scratch each step.")
print("The KV cache is what keeps decode linear.")

## Experiments: Try These

1. **Temperature sweep**: In the sampling demo, change the temperature from 0 to 2.0 in steps of 0.25 and plot the entropy of the resulting distribution. At what temperature does the distribution become nearly uniform? What does uniform sampling imply for output quality?

2. **Grouped Query Attention (GQA) memory savings**: Modify `kv_cache_memory_gb()` to accept separate `num_q_heads` and `num_kv_heads` parameters. For Llama-3-70B (64 Q-heads, 8 KV-heads), compute the memory savings vs Multi-Head Attention (64 KV-heads). Plot how the savings scale from MHA → GQA → MQA (Multi-Query, 1 KV-head).

3. **Real tokenizer comparison**: Install `pip install tiktoken transformers` and compare token counts for the same 10 sentences across GPT-4's tokenizer (`cl100k_base`, 100k vocab) and Llama-3's tokenizer (`llama3`, 128k vocab). Which is more efficient on code vs prose?

## Key Takeaways

- **Autoregressive generation is sequential by design**: each token depends on all prior tokens, so decode cannot be parallelized across output positions — this is the fundamental throughput limit of LLMs.
- **The KV cache converts $O(n^2)$ attention into $O(n)$ decode**: by storing K and V tensors from prefill and growing them incrementally, each decode step only needs to compute Q for the new token and do one cache lookup.
- **Prefill is compute-bound, decode is memory-bandwidth-bound**: this asymmetry drives hardware selection (decode needs HBM bandwidth), batching strategy (batch more to amortize decode memory reads), and disaggregation (run prefill and decode on separate hardware).
- **Sampling strategy is a real engineering decision**: temperature, top-k, and top-p control quality/diversity tradeoffs at the cost of only a few microseconds — get them wrong and you get repetitive or incoherent outputs regardless of model quality.
- **Next up — Day 02**: Transformer Blocks & Attention Deep Dive — the Q, K, V projection matrices, multi-head attention, and the feed-forward sublayer in detail.

## References

- *Inference Engineering* Ch 2.2 (pp. 46–54) — Philip Kiely, Baseten Books 2026
- Vaswani et al. (2017), "Attention Is All You Need" — original transformer paper
- Dao et al. (2022), "FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness"